# MERFISH HCC

In this notebook we demonstrate how mosna can be used to analyze spatiallly resolved omics data.  
The data used is from the publication by [Magen et al., Nature Medecine, 2023](https://doi.org/10.1038/s41591-023-02345-0) "Intratumoral dendritic cell–CD4+ T helper cell niches enable CD8+ T cell differentiation following PD-1 blockade in hepatocellular carcinoma".  
Here 6 tumors of hepatocellular carcinoma (HCC) were processed with the the MERFISH method (Vizgen platform) to produce maps of 400 transcripts and study the relationship between cell types in tumors with respect to responde to anti-PD-1.  

## Imports and data loading

In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning, FitFailedWarning
warnings.simplefilter('ignore', FitFailedWarning)
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', FutureWarning)
warnings.simplefilter('ignore', DeprecationWarning)
warnings.simplefilter('ignore', UserWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import re
import os
from time import time
import warnings
import joblib
import json
from pathlib import Path
from time import time
from tqdm import tqdm
import copy
import matplotlib as mpl
import napari
import colorcet as cc
import composition_stats as cs
from sklearn.impute import KNNImputer
from lifelines import KaplanMeierFitter, CoxPHFitter

from tysserand import tysserand as ty
from mosna import mosna

import matplotlib as mpl
mpl.rcParams["figure.facecolor"] = 'white'
mpl.rcParams["axes.facecolor"] = 'white'
mpl.rcParams["savefig.facecolor"] = 'white'

In [2]:
# If need to reload modules after their modification
from importlib import reload
ty = reload(ty)
mosna = reload(mosna)

In [3]:
RUN_LONG = True

### Objects data

Load files that contains all the detected objects (the cells) across all samples and clinical data.  
Omics is available here: https://zenodo.org/records/7758080  
Clinical data is available here: https://www.nature.com/articles/s41591-023-02345-0/tables/1

In [4]:
# data_dir = Path("../data/raw/MERFISH_HCC_Magen_2023")
# objects_path = data_dir / "SingleCells.csv"

# if objects_path.with_suffix('.parquet').exists():
#     obj = pd.read_parquet(objects_path.with_suffix('.parquet'))
# else:
#     obj = pd.read_csv(objects_path)
#     # for latter use
#     obj.to_parquet(objects_path.with_suffix('.parquet'))
# obj

In [5]:
data_dir = Path("../data/raw/MERFISH_HCC_Magen_2023")
omics_dir = data_dir / 'omics_data'

In [6]:
# list all files with patient and sample id

pattern = "*_region_*_cell_by_gene.csv"
files = glob.glob(os.path.join(omics_dir, pattern))

all_paths = []
for filepath in files:
    filename = os.path.basename(filepath)
    # Use regex to extract the two numbers
    match = re.search(r'^(\d+)_region_(\d+)_cell_by_gene\.csv$', filename)
    if match:
        first_num = int(match.group(1))  # Number before '_region'
        second_num = int(match.group(2))  # Number before '_cell'
        all_paths.append((filepath, first_num, second_num))

In [7]:
rnas = pd.read_csv(all_paths[0][0], index_col=0, nrows=0)
all_omics = rnas.columns.values
marker_cols = [x for x in all_omics if not x.startswith('Blank')]
blank_cols = [x for x in all_omics if x.startswith('Blank')]

# reorder columns to make sure we save them with the same order for all samples
all_omics = marker_cols + blank_cols

print(f"There are {len(marker_cols)} transcript columns and {len(blank_cols)} blank columns")

There are 400 transcript columns and 61 blank columns


In [8]:
processed_dir = Path('../data/processed/MERFISH_HCC_Magen_2023')
processed_dir.mkdir(parents=True, exist_ok=True)
pos_cols = ['center_x', 'center_y']

In [ ]:
for file_path, patient_id, sample_id in tqdm(all_paths):
    rnas = pd.read_csv(file_path, index_col=0, dtype='int')
    rnas.reset_index(drop=True, inplace=True)
    path_metadata = omics_dir / f"{patient_id}_region_{sample_id}_cell_metadata.csv"
    metadata = pd.read_csv(path_metadata, usecols=pos_cols)
    metadata.reset_index(drop=True, inplace=True)
    cell_data = pd.concat([metadata, rnas[all_omics]], axis=1)
    cell_data.to_parquet(processed_dir / f"nodes_patient-{patient_id}_sample-{sample_id}.parquet")

del rnas, metadata, cell_data

100%|██████████| 10/10 [00:42<00:00,  4.30s/it]


In [9]:
data_index = [(x[1], x[2]) for x in all_paths]
data_index

[(1014, 1),
 (122, 1),
 (1012, 0),
 (1017, 0),
 (63, 0),
 (1029, 0),
 (122, 0),
 (1003, 0),
 (1014, 0),
 (1012, 1)]

In [40]:
del_zero_cells = False
del_lower_counts = True
lower_counts = 5
del_upper_counts = True
upper_counts = 1250
del_lower_uniq_genes = True
lower_uniq_genes = 5
# del_lower_area not implemented
# del_upper_area not implemented
del_blanks = True
# MERFISH data, contrary to scRNA-seq, doesn't need normalization of counts
normalize_counts = False
log_normalize = True

nodes_dir = processed_dir / "preprocessed"
nodes_dir.mkdir(parents=True, exist_ok=True)

# save preprocessing metadata
preproc_metadata = {
    'del_zero_cells': del_zero_cells,
    'del_lower_counts': del_lower_counts,
    'lower_counts': lower_counts,
    'del_upper_counts': del_upper_counts,
    'del_lower_uniq_genes': del_lower_uniq_genes,
    'lower_uniq_genes': lower_uniq_genes,
    'upper_counts': upper_counts,
    'del_blanks': del_blanks,
    'normalize_counts': normalize_counts,
    'log_normalize': log_normalize,
}
with open(nodes_dir / "preproc_metadata.json", "w") as fp:
    json.dump(preproc_metadata , fp, indent = 4)

# actually preprocess data
for patient_id, sample_id in data_index:
    print(f"patient {patient_id} sample {sample_id}:")
    data = pd.read_parquet(processed_dir / f"nodes_patient-{patient_id}_sample-{sample_id}.parquet")

    # delete cells with zero counts
    if del_zero_cells:
        counts = data[marker_cols].sum(axis=1)
        select = counts != 0
        n_delete = len(data) - select.sum()
        prop_delete = n_delete / len(data) * 100
        print(f"    {n_delete} cells with 0 counts ({prop_delete:.3g}%)")
        data = data.loc[select, :]
        # reindex data
        data.reset_index(drop=True, inplace=True)
    
    if del_lower_counts:
        counts = data[marker_cols].sum(axis=1)
        select = counts > lower_counts
        n_delete = len(data) - select.sum()
        prop_delete = n_delete / len(data) * 100
        print(f"    {n_delete} cells with counts <= {lower_counts} ({prop_delete:.3g}%)")
        data = data.loc[select, :]
        # reindex data
        data.reset_index(drop=True, inplace=True)
    
    if del_upper_counts:
        counts = data[marker_cols].sum(axis=1)
        select = counts < upper_counts
        n_delete = len(data) - select.sum()
        prop_delete = n_delete / len(data) * 100
        print(f"    {n_delete} cells with counts >= {upper_counts} ({prop_delete:.3g}%)")
        data = data.loc[select, :]
        # reindex data
        data.reset_index(drop=True, inplace=True)
    
    if del_lower_uniq_genes:
        count_genes = np.sum(data[marker_cols] != 0, axis=1)
        select = count_genes > lower_uniq_genes
        n_delete = len(data) - select.sum()
        prop_delete = n_delete / len(data) * 100
        print(f"    {n_delete} cells with unique genes <= {lower_uniq_genes} ({prop_delete:.3g}%)")
        data = data.loc[select, :]
        # reindex data
        data.reset_index(drop=True, inplace=True)

    # discard blank columns (useful only for QC)
    if del_blanks:
        data = data[pos_cols + marker_cols]

    # normalize to 10,000 counts
    if normalize_counts:
        data.loc[:, marker_cols] = data.loc[:, marker_cols].div(data.loc[:, marker_cols].sum(axis=1), axis=0) * 1e4
    
    # log-normalize counts
    if log_normalize:
        data.loc[:, marker_cols] = np.log1p(data.loc[:, marker_cols])
    
    data.to_parquet(nodes_dir / f"nodes_patient-{patient_id}_sample-{sample_id}.parquet")

patient 1014 sample 1:
    2911 cells with counts <= 5 (11.4%)
    0 cells with counts >= 1250 (0%)
    377 cells with unique genes <= 5 (1.67%)
patient 122 sample 1:
    34253 cells with counts <= 5 (22.6%)
    1 cells with counts >= 1250 (0.000852%)
    2288 cells with unique genes <= 5 (1.95%)
patient 1012 sample 0:
    3381 cells with counts <= 5 (1.19%)
    180 cells with counts >= 1250 (0.0643%)
    257 cells with unique genes <= 5 (0.0918%)
patient 1017 sample 0:
    2326 cells with counts <= 5 (3.79%)
    2 cells with counts >= 1250 (0.00339%)
    224 cells with unique genes <= 5 (0.379%)
patient 63 sample 0:
    45017 cells with counts <= 5 (19.7%)
    0 cells with counts >= 1250 (0%)
    3891 cells with unique genes <= 5 (2.12%)
patient 1029 sample 0:
    2610 cells with counts <= 5 (2.17%)
    34 cells with counts >= 1250 (0.0289%)
    212 cells with unique genes <= 5 (0.18%)
patient 122 sample 0:
    5682 cells with counts <= 5 (6.19%)
    25 cells with counts >= 1250 (0.02

In [41]:
#  check that there is no NA
for patient_id, sample_id in data_index:
    data = pd.read_parquet(nodes_dir / f"nodes_patient-{patient_id}_sample-{sample_id}.parquet")
    n_na = data.isna().sum().sum()
    if n_na != 0:
        print(f"patient {patient_id} sample {sample_id}: {n_na} NAs")
        break
print("All data have no NA")

All data have no NA


### Survival data

In [42]:
survival_path = data_dir / "response.ods"
surv = pd.read_excel(survival_path, index_col=0)
surv

,Response,Etiology,Age,Sex,Ethnicity,Necrosis (%),Immune infiltration,Drug,
x,,,,,,,,,
15,Nonresponder,Hep B,41,Male,Asian,0,Exclusion,Nivolumab,
57,Nonresponder,Hep C + NASH,61,Male,White,0,Exclusion,Nivolumab,
63,Responder,Hep C + NASH,63,Male,White,100,High,Nivolumab,
71,Nonresponder,Hep B,29,Male,Asian,15,Low,Nivolumab,
98,Nonresponder,Hep B,56,Female,Asian,0,Exclusion,Nivolumab,
104,Nonresponder,Hep B,35,Male,African American,0,Exclusion,Nivolumab,
106,Responder,Hep C,66,Female,African American,100,High,Nivolumab,
121,Nonresponder,Hep B,38,Male,Asian,0,High,Nivolumab,
124,Nonresponder,Hep C,65,Male,White,0,Low,Cemiplimab,


### All samples network reconstruction

In [43]:
dir_fig_save = processed_dir / 'figures'

In [ ]:
min_neighbors = 0
edges_dir = nodes_dir

# or save in separate file for convenience
if RUN_LONG:
    for patient_id, sample_id in tqdm(data_index):
        nodes = data = pd.read_parquet(nodes_dir / f"nodes_patient-{patient_id}_sample-{sample_id}.parquet")
        coords = nodes[pos_cols].values
        pairs = ty.build_delaunay(coords)

        if min_neighbors > 0:
            pairs = ty.link_solitaries(
                coords, 
                pairs, 
                min_neighbors=min_neighbors,
                )

        edges = pd.DataFrame(data=pairs, columns=['source', 'target'])
        edges.to_parquet(edges_dir / f'edges_patient-{patient_id}_sample-{sample_id}.parquet', index=False)

### Batch correction

#### make UMAP before batch correction

In [ ]:
nodes_agg = mosna.aggregate_nodes(
    nodes_dir=nodes_dir,
    use_cols=marker_cols,
)

In [ ]:
embed_viz, _ = mosna.get_reducer(nodes_agg[marker_cols], nodes_dir)

In [ ]:
fig_step = '01_processing'

In [ ]:
fig, ax, color_mapper = mosna.plot_clusters(
    embed_viz, 
    cluster_labels=nodes_agg['patient'], 
    save_dir=None,
    return_cmap=True,
    show_id=False,
    )
plt.savefig(save_dir / f'{fig_step}-UMAP_before_batch_correction_per_patient.jpg', bbox_inches='tight', facecolor='white')

In [ ]:
fig, ax, color_mapper = mosna.plot_clusters(
    embed_viz, 
    cluster_labels=nodes_agg['sample'], 
    save_dir=None,
    return_cmap=True,
    show_id=False,
    )
plt.savefig(save_dir / f'{fig_step}-UMAP_before_batch_correction_per_sample.jpg', bbox_inches='tight', facecolor='white')

#### Perform batch correction

In [ ]:
nodes_dir, nodes_corr = mosna.batch_correct_nodes(
    nodes_dir=nodes_dir,
    use_cols=marker_cols,
    batch_key='patient',
    return_nodes=True,
    )

In [ ]:
embed_viz, _ = mosna.get_reducer(nodes_corr[marker_cols], nodes_dir, random_state=0)

Computing dimensionality reduction


/home/alexis/miniconda3/envs/test_spatialomics/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
fig, ax, color_mapper = mosna.plot_clusters(
    embed_viz, 
    cluster_labels=nodes_corr['patient'], 
    save_dir=None,
    return_cmap=True,
    show_id=False,
    )
plt.savefig(save_dir / f'{fig_step}-UMAP_after_batch_correction_per_patient.jpg', bbox_inches='tight', facecolor='white')